# Module D — Asymmetric Fusion Rule Sweep (FROZEN E/D)

Protocol:
- Load pretrained `CDDFuse_MIF.pth` (E + D từ paper repo)
- **FREEZE Encoder + Decoder** — chỉ train Fusion modules
- **Phase II ONLY** (skip Phase I) — 30 epoch trên 738 cặp Harvard
- Baseline so sánh = CDDFuse paper (Sum-Sum)

Variant chọn từ `variants/registry_asymmetric.py` — 1 trong:
- Stage 1 D-Base sweep: `AsymD-DB{0..6}-Base{Sum,Gated,L1Norm,VSM,LocalEnergy,LocalEntropy,WeightedAvg}`
- Stage 2 D-Detail sweep: `AsymD-DD{0..8}-Detail{Sum,Gated,MaxAbs,Saliency,SF,LocalEnergy,SML,L1Norm,PCNNsoft}`

Datasets: cùng 738 cặp train + 72 cặp test như AG-45ep.
Tốc độ kỳ vọng: ~2-3h/variant trên P100 (4× nhanh hơn full train do E/D frozen).

## Cell 1 â€” Config

In [ ]:
# Chọn 1 variant từ VARIANT_REGISTRY_ASYM (xem variants/registry_asymmetric.py)
VARIANT     = 'AsymD-DB3-BaseVSM'   # đổi tên này khi train variant khác
REPO_URL    = 'https://github.com/kienvbhp872004/Image-Fusion.git'
REPO_BRANCH = 'main'
EPOCHS      = 30    # Phase II only (đã skip Phase I)
BATCH       = 8
SEED        = 42
USE_AMP     = True

PRETRAINED_DATASET = 'cddfuse-pretrained'

import os, subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
GPU_NAME = r.stdout.strip().splitlines()[0] if r.stdout else 'unknown'
print(f'[gpu] {GPU_NAME}')
print(f'[variant] {VARIANT}  epochs={EPOCHS}  batch={BATCH}')

## Cell 2 â€” Clone repo, install deps, downgrade torch cho P100 sm_60

In [ ]:
!git clone --branch $REPO_BRANCH $REPO_URL /kaggle/working/Image-Fusion
%cd /kaggle/working/Image-Fusion
!pip install -q einops==0.4.1 kornia==0.6.12 h5py tqdm scikit-image

# Downgrade torch Ä‘á»ƒ support P100 sm_60 (torch 2.6+ drop Pascal). 2.5.1 lÃ  báº£n cuá»‘i.
!pip uninstall -y torch torchvision torchaudio 2>&1 | tail -3
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!python -c "import torch; print('[torch]', torch.__version__, 'cuda:', torch.cuda.is_available(), 'cap:', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'N/A')"

## Cell 3 â€” Stage Harvard medical vá»›i split train/test cÃ³ sáºµn (MyDatasets/)

In [ ]:
import shutil, pathlib, glob

def find_dataset_root(slug):
    for c in [f'/kaggle/input/{slug}', f'/kaggle/input/datasets/kienvbhp1234/{slug}']:
        if pathlib.Path(c).exists(): return pathlib.Path(c)
    m = glob.glob(f'/kaggle/input/**/{slug}', recursive=True)
    if m: return pathlib.Path(m[0])
    raise FileNotFoundError(slug)

TRAIN_SRC = find_dataset_root('harvard-medical-train')   # 738 cáº·p train (160+245+333)
TEST_SRC  = find_dataset_root('harvard-medical-fusion')  # 72 cáº·p test (24Ã—3)
print(f'[paths] train={TRAIN_SRC}\n[paths] test ={TEST_SRC}')

# Stage thÃ nh cáº¥u trÃºc MyDatasets/{modal}/{train,test}/{src,MRI}/
POOL = pathlib.Path('/kaggle/working/Image-Fusion/Havard-Medical-Image-Fusion-Datasets-main/Havard-Medical-Image-Fusion-Datasets-main/MyDatasets')
POOL.mkdir(parents=True, exist_ok=True)

for modal in ['CT-MRI', 'PET-MRI', 'SPECT-MRI']:
    sub = modal.split('-')[0]
    # Train: copy tá»« harvard-medical-train
    train_src = TRAIN_SRC / modal
    if train_src.exists():
        dst = POOL / modal / 'train'
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copytree(train_src, dst, dirs_exist_ok=True)
    # Test: copy tá»« harvard-medical-fusion
    test_src = TEST_SRC / modal
    if test_src.exists():
        dst = POOL / modal / 'test'
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copytree(test_src, dst, dirs_exist_ok=True)
    n_train = len(list((POOL / modal / 'train' / sub).glob('*.png')))
    n_test  = len(list((POOL / modal / 'test'  / sub).glob('*.png')))
    print(f'[stage] {modal}: train={n_train} test={n_test}')

## Cell 4 â€” Pre-process: extract patches â†’ h5

In [ ]:
%cd /kaggle/working/Image-Fusion/models/MMIF-CDDFuse
!python dataprocessing_MIF.py

## Cell 5 — Train asymmetric (E/D frozen, Phase II only, 30 ep)

In [ ]:
import glob, pathlib
candidates = glob.glob(f'/kaggle/input/**/CDDFuse_MIF.pth', recursive=True)
assert candidates, 'CDDFuse_MIF.pth không tìm thấy trong /kaggle/input/* — attach dataset cddfuse-pretrained'
PRETRAINED = candidates[0]
print(f'[pretrained] {PRETRAINED}')

amp_flag = '--amp' if USE_AMP else ''
!python train_asymmetric.py \n    --variant     $VARIANT \n    --pretrained  $PRETRAINED \n    --num_epochs  $EPOCHS \n    --batch       $BATCH \n    --coeff_decomp 2.0 \n    --seed        $SEED \n    --output      /kaggle/working/ \n    $amp_flag

## Cell 6 — Inference + per-image metrics

In [ ]:
import glob
ckpts = sorted(glob.glob(f'/kaggle/working/CDDFuse-{VARIANT}_MIF_*.pth'))
assert ckpts, 'No checkpoint found from Cell 5'
CKPT = ckpts[-1]
OUT_DIR = f'/kaggle/working/CDDFuse-{VARIANT}'
print(f'[ckpt] {CKPT}')

import shutil, pathlib
MYDS = pathlib.Path('/kaggle/working/Image-Fusion/Havard-Medical-Image-Fusion-Datasets-main/Havard-Medical-Image-Fusion-Datasets-main/MyDatasets')
TEST_OUT = pathlib.Path('/kaggle/working/Image-Fusion/data/reference')
for modal in ['CT-MRI', 'PET-MRI', 'SPECT-MRI']:
    sub = modal.split('-')[0]
    src_test = MYDS / modal / 'test'
    if not src_test.exists():
        print(f'[skip] {modal} test missing'); continue
    (TEST_OUT / modal).mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_test, TEST_OUT / modal, dirs_exist_ok=True)
    n = len(list((TEST_OUT / modal / sub).glob('*.png')))
    print(f'[test ] {modal}: {n} cặp staged')

for modal in ['CT', 'PET', 'SPECT']:
    !python evaluate_cddfuse.py \n        --variant       $VARIANT \n        --modal         $modal \n        --ckpt          $CKPT \n        --harvard_root  /kaggle/working/Image-Fusion/data/reference \n        --out_dir       $OUT_DIR \n        --save_perimage

## Cell 7 â€” Package output

In [ ]:
import tarfile, json, hashlib, datetime, os, glob
h = hashlib.sha256()
with open(CKPT, 'rb') as f:
    for chunk in iter(lambda: f.read(8192), b''): h.update(chunk)
stamp = {
    'variant_name': f'CDDFuse-{VARIANT}',
    'based_on':     'Module D — Asymmetric Fusion Rule (E/D frozen + Phase II only)',
    'datetime':     datetime.datetime.utcnow().isoformat() + 'Z',
    'train_mode':   'phase2_only_frozen_ed',
    'gpu':          GPU_NAME,
    'epochs':       EPOCHS,
    'batch':        BATCH,
    'amp':          USE_AMP,
    'seed':         SEED,
    'pretrained':   os.path.basename(PRETRAINED),
    'coeff_decomp': 2.0,
    'ckpt_sha256':  h.hexdigest(),
}
with open(f'{OUT_DIR}/_ablation_stamp.json', 'w') as f:
    json.dump(stamp, f, indent=2)

tar_path = f'/kaggle/working/CDDFuse-{VARIANT}_results.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(OUT_DIR, arcname=f'CDDFuse-{VARIANT}')
    tar.add(CKPT, arcname=f'CDDFuse-{VARIANT}.pth')
    hist_files = glob.glob(f'/kaggle/working/CDDFuse-{VARIANT}_MIF_*_train_history.json')
    for hf in hist_files:
        tar.add(hf, arcname=os.path.basename(hf))
print('[done]', tar_path)